In [1]:
#All packages needed to run TwINFER simulation and inference are listed here. 
#If any of them are not installed, please install them using pip or conda env.
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numba
import tqdm
import scipy
import seaborn
import os
import sys
import joblib
from itertools import product
from pathlib import Path



In [2]:
path_to_simulations = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_3_simulations/"
path_to_code_repo = "/home/gzu5140/Keerthana_b1042/grnInference/code/TwINFER/"
path_to_plot_data = "/home/gzu5140/Keerthana_b1042/grnInference/plot_data/figure_3_five_gene/"
os.makedirs(path_to_plot_data, exist_ok=True)

## Importing necessary functions

In [3]:
# Calculation functions
import sys
sys.path.append(str(path_to_code_repo))
import importlib
from TwINFER_function_scripts import correlation_analysis_functions
from TwINFER_function_scripts import correlation_analysis_helpers
from TwINFER_function_scripts import infer_with_twinfer

importlib.reload(correlation_analysis_functions)
importlib.reload(correlation_analysis_helpers)
importlib.reload(infer_with_twinfer)

from TwINFER_function_scripts.correlation_analysis_functions import (
    generate_random_shuffle
    # calculate_pairwise_gene_gene_correlation_matrix,
    # check_system_in_steady_state,
    # check_gene_gene_correlation_threshold,
    # calculate_twin_random_pair_correlations,
    # differentiate_single_state_reg_and_multiple_states,
    # identify_reg_if_multiple_states,
    # get_cross_correlations,
    # identify_actual_directed_edges
)

# Helper functions
from TwINFER_function_scripts.correlation_analysis_helpers import (
    extract_param_index,
    read_input_matrix,
    split_and_merge_simulations,
    get_param_data, 
    plot_matrix_as_heatmap,
    print_summary,
    plot_network
)

from TwINFER_function_scripts.infer_with_twinfer import (
    infer_with_twinfer
)

In [10]:
# Network types (folder names)
network_types = [
    "A_B",
    "A_rep_B", 
    "A_rep_B_B_rep_A",
    "A_rep_B_B_to_A",
    "A_to_B",
    "A_to_B_B_to_A"
]

# Base configuration template
base_config_template = {
    'n_cells': 6000,
    'simulation_time_before_division': 1000,
    'twin_simulation_time_after_division': 48,
    'twin_measurement_resolution': 1,
    "path_to_connectivity_matrix": "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/interaction_matrix_A_to_B.txt",  # Will be updated per network type
    "param_csv": "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/median_param.csv",  # Will be updated per network type
    "rows_to_use": [[0,1]],
    "path_to_plot_data": path_to_plot_data,
    "log_file": None,  # Will be updated per network type
    "type": None,  # Will be updated per network type
    "number_of_parallel_parameters": 1,
    "number_of_cores_per_parameter": 18,
}

# Time points for analysis
t1, t2 = 1, 20  # Update these as needed

# Process each network type
all_correlation_matrices = {}

for network_type in network_types:
    print(f"Processing network type: {network_type}")
    
    # Find simulation file for this network type
    network_folder = os.path.join(path_to_simulations, network_type)
    if not os.path.exists(network_folder):
        print(f"Warning: Network folder not found: {network_folder}")
        continue
    
    # Look for CSV files in the network type folder
    csv_files = [f for f in Path(network_folder).glob("*.csv") 
                 if f.name.startswith("df") and "rep_5" in f.name]
    if not csv_files:
        print(f"Warning: No CSV files starting with 'df' and containing 'rep_1' found in {network_folder}")
        continue
    
    # Use the first CSV file found (or you can add logic to select specific one)
    path_to_simulation_file = str(csv_files[0])
    print(f"Using simulation file: {os.path.basename(path_to_simulation_file)}")
    
    # Update config for this network type
    config = base_config_template.copy()
    config["type"] = network_type
    config["log_file"] = f"{path_to_code_repo}/example_simulation_output/{network_type}_log.jsonl"
    
    # Check if required files exist
    if not os.path.exists(config["path_to_connectivity_matrix"]):
        print(f"Warning: Connectivity matrix not found for {network_type}")
        continue
    if not os.path.exists(config["param_csv"]):
        print(f"Warning: Parameter CSV not found for {network_type}")
        continue
    
    try:
        # Run inference for this network type
        correlation_matrices = infer_with_twinfer(
                path_to_simulation_file = path_to_simulation_file, 
                base_config = config, 
                t1 = t1, t2 = t2,
                check_for_steady_state=False, 
                plot_correlation_matrices_as_heatmap=True, 
                have_any_output=True,
                infer_direction_for_which_edges = "all-edges",
                merge_to_multiple_states  = False,
                match_sim_details = False


            )
            
            # Store the correlation matrices
        all_correlation_matrices[network_type] = correlation_matrices
            
            # Save the directional correlation matrix for this network type
        correlation_file_name = f"{path_to_plot_data}/filtered_directional_correlation_type_{network_type}.csv"
        correlation_matrices['direction_matrix'].to_csv(correlation_file_name)            
        print(f"Successfully processed {network_type}")
        print(f"Saved correlation matrix to: {correlation_file_name}")
        
    except Exception as e:
        print(f"Error processing {network_type}: {str(e)}")
        continue


Processing network type: A_B
Using simulation file: df_rows_0_1_08082025_093558_ncells_6000_A_B_rep_5.csv
Could not ascertain corresponding parameter rows to check for gene parameters


KeyboardInterrupt: 

## 5-gene cascade

In [10]:
import os
import pandas as pd
from pathlib import Path
# Base configuration template
base_config_template = {
    'n_cells': 6000,
    'simulation_time_before_division': 1000,
    'twin_simulation_time_after_division': 24,
    'twin_measurement_resolution': 1,
    "path_to_connectivity_matrix": f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_5_gene_linear_cascade.txt",  # Will be updated per network type
    "param_csv":f"{path_to_code_repo}/simulation_example_input_data/median_parameter.csv",  # Will be updated per network type
    "rows_to_use": [[0,0,0,0,0]],
    "path_to_plot_data": path_to_plot_data,
    "log_file": None,  # Will be updated per network type
    "type": None,  # Will be updated per network type
    "number_of_parallel_parameters": 1,
    "number_of_cores_per_parameter": 18,
    "seed" : 101010
}

# Time points for analysis
t1, t2 = 1, 20  # Update these as needed

# Process each network type
all_correlation_matrices = {}
   
# Use the first CSV file found (or you can add logic to select specific one)
path_to_simulation_file = f"/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/testing_hill/df_rows_0_0_0_0_0_03122025_221647_ncells_6000_Five_gene_cascade_fixed_K_0_7_6a281f3f.csv"
print(f"Using simulation file: {os.path.basename(path_to_simulation_file)}")
    
    # Update config for this network type
config = base_config_template.copy()
network_type = "five_gene_linear_cascade" 
config["type"] = {network_type}
config["log_file"] = f"{path_to_code_repo}/example_simulation_output/{network_type}_log.jsonl"
    
# Check if required files exist
if not os.path.exists(config["path_to_connectivity_matrix"]):
    print(f"Warning: Connectivity matrix not found for {network_type}")

if not os.path.exists(config["param_csv"]):
    print(f"Warning: Parameter CSV not found for {network_type}")
    
try:
    # Run inference for this network type
    correlation_matrices = infer_with_twinfer(
            path_to_simulation_file = path_to_simulation_file, 
            base_config = config, 
            t1 = t1, t2 = t2,
            check_for_steady_state=False, 
            plot_correlation_matrices_as_heatmap=True, 
            have_any_output=True,
            remove_twin_structure = True,
            infer_direction_for_which_edges = "single-state", seed = 2025
        )
        
        # Store the correlation matrices
    all_correlation_matrices[network_type] = correlation_matrices
        
        # Save the directional correlation matrix for this network type
    correlation_file_name = f"{path_to_plot_data}/filtered_directional_correlation_type_{network_type}_random.csv"
    correlation_matrices['direction_matrix'].to_csv(correlation_file_name)
            
    print(f"Successfully processed {network_type}")
    print(f"Saved correlation matrix to: {correlation_file_name}")
except Exception as e:
    print(f"Error processing {network_type}: {str(e)}")

Using simulation file: df_rows_0_0_0_0_0_03122025_221647_ncells_6000_Five_gene_cascade_fixed_K_0_7_6a281f3f.csv
{'k_on_gene_1': 0.66, 'k_off_gene_1': 8.6, 'mrna_half_life_gene_1': 4, 'protein_half_life_gene_1': 45, 'k_prod_protein_gene_1': 560, 'k_prod_mRNA_gene_1': 2, 'k_deg_mRNA_gene_1': np.float64(0.17328679513998632), 'k_deg_protein_gene_1': np.float64(0.015403270679109895), 'k_on_gene_2': 0.66, 'k_off_gene_2': 8.6, 'mrna_half_life_gene_2': 4, 'protein_half_life_gene_2': 45, 'k_prod_protein_gene_2': 560, 'k_prod_mRNA_gene_2': 2, 'k_deg_mRNA_gene_2': np.float64(0.17328679513998632), 'k_deg_protein_gene_2': np.float64(0.015403270679109895), 'k_on_gene_3': 0.66, 'k_off_gene_3': 8.6, 'mrna_half_life_gene_3': 4, 'protein_half_life_gene_3': 45, 'k_prod_protein_gene_3': 560, 'k_prod_mRNA_gene_3': 2, 'k_deg_mRNA_gene_3': np.float64(0.17328679513998632), 'k_deg_protein_gene_3': np.float64(0.015403270679109895), 'k_on_gene_4': 0.66, 'k_off_gene_4': 8.6, 'mrna_half_life_gene_4': 4, 'protein_h

## Network in figure 1

In [ ]:
import os
import pandas as pd
from pathlib import Path
# Base configuration template
base_config_template = {
    'n_cells': 6000,
    'simulation_time_before_division': 1000,
    'twin_simulation_time_after_division': 48,
    'twin_measurement_resolution': 1,
    "path_to_connectivity_matrix": f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_figure_1_network.txt",  # Will be updated per network type
    "param_csv":f"{path_to_code_repo}/simulation_example_input_data/median_parameter.csv",  # Will be updated per network type
    "rows_to_use": [[0]*14],
    "path_to_plot_data": path_to_plot_data,
    "log_file": None,  # Will be updated per network type
    "type": None,  # Will be updated per network type
    "number_of_parallel_parameters": 1,
    "number_of_cores_per_parameter": 18,
    "seed" : 101010
}

# Time points for analysis
t1, t2 = 1, 20  # Update these as needed

# Process each network type
all_correlation_matrices = {}
   
# Use the first CSV file found (or you can add logic to select specific one)
path_to_simulation_file = f"{path_to_simulations}/df_rows_0_0_0_0_0_0_0_0_0_0_0_0_0_0_16112025_110505_ncells_6000_Figure1Network_0_0_718b0813.csv"
print(f"Using simulation file: {os.path.basename(path_to_simulation_file)}")
    
    # Update config for this network type
config = base_config_template.copy()
network_type = "five_gene_linear_cascade" 
config["type"] = {network_type}
config["log_file"] = f"{path_to_code_repo}/example_simulation_output/{network_type}_log.jsonl"
    
# Check if required files exist
if not os.path.exists(config["path_to_connectivity_matrix"]):
    print(f"Warning: Connectivity matrix not found for {network_type}")

if not os.path.exists(config["param_csv"]):
    print(f"Warning: Parameter CSV not found for {network_type}")
    
try:
    # Run inference for this network type
    correlation_matrices = infer_with_twinfer(
            path_to_simulation_file = path_to_simulation_file, 
            base_config = config, 
            t1 = t1, t2 = t2,
            check_for_steady_state=False, 
            plot_correlation_matrices_as_heatmap=True, 
            have_any_output=True,
            infer_direction_for_which_edges = "single-state",
        )
        
        # Store the correlation matrices
    all_correlation_matrices[network_type] = correlation_matrices
        
        # Save the directional correlation matrix for this network type
    correlation_file_name = f"{path_to_plot_data}/filtered_directional_correlation_type_{network_type}.csv"
    correlation_matrices['direction_matrix'].to_csv(correlation_file_name)
            
    print(f"Successfully processed {network_type}")
    print(f"Saved correlation matrix to: {correlation_file_name}")
        
except Exception as e:
    print(f"Error processing {network_type}: {str(e)}")